In [31]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-local-app")
    .master("local[*]")
    .getOrCreate()
)

In [32]:
from datetime import datetime, timezone
from typing import Any

from pyspark.sql import Row

customers: list[dict[str, Any]] = [
    {
        "id": 1,
        "first_name": "Иван",
        "last_name": "Иванов",
        "email": "ivanov@example.com",
        "phone_numbers": Row(mobile="+7 (999) 111-22-33", home="+7 (495) 111-22-33"),
        "is_customer": True,
        "amount_paid": 15450.50,
        "customer_from": datetime(2023, 5, 12, 10, 30, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2026, 1, 15, 14, 22, tzinfo=timezone.utc),
        "courses": [1, 2, 3]
    },
    {
        "id": 2,
        "first_name": "Анна",
        "last_name": "Петрова",
        "email": "petrova@example.com",
        "phone_numbers": Row(mobile="+7 (999) 141-22-33", home="+7 (495) 111-12-33"),
        "is_customer": False,
        "amount_paid": 0.00,
        "customer_from": None,
        "last_updated_ts": datetime(2026, 2, 20, 9, 15, tzinfo=timezone.utc),
        "courses": [1, 3]
    },
    {
        "id": 3,
        "first_name": "Dmitry",
        "last_name": "Smith",
        "email": "d.smith@global-tech.com",
        "phone_numbers": Row(mobile="+7 (989) 111-12-33", home="+7 (465) 111-21-33"),
        "is_customer": True,
        "amount_paid": 450300.00,
        "customer_from": datetime(2021, 11, 1, 8, 0, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2026, 3, 10, 18, 45, tzinfo=timezone.utc),
        "courses": [1, 2]
    },
    {
        "id": 4,
        "first_name": "Елена",
        "last_name": "Сидорова",
        "email": "elena.sid@yandex.ru",
        "phone_numbers": Row(mobile="+7 (989) 110-12-33", home="+7 (465) 111-29-33"),
        "is_customer": True,
        "amount_paid": 1200.00,
        "customer_from": datetime(2025, 8, 24, 19, 12, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2025, 8, 24, 19, 30, tzinfo=timezone.utc),
        "courses": [1]
    },
    {
        "id": 5,
        "first_name": "Алексей",
        "last_name": "Уваров",
        "email": "uvarov_alex@mail.ru",
        "phone_numbers": Row(mobile="+7 (989) 111-02-33", home="+7 (465) 511-21-33"),
        "is_customer": False,
        "amount_paid": 8900.75,
        "customer_from": datetime(2024, 2, 10, 11, 0, tzinfo=timezone.utc),
        "last_updated_ts": datetime(2026, 4, 1, 12, 0, tzinfo=timezone.utc),
        "courses": [1, 2, 3]
    },
]

In [33]:
import pandas as pd

In [34]:
df_customers = spark.createDataFrame([Row(**customer) for customer in customers])

In [35]:
df_customers.select("id", "first_name", "last_name").show()

+---+----------+---------+
| id|first_name|last_name|
+---+----------+---------+
|  1|      Иван|   Иванов|
|  2|      Анна|  Петрова|
|  3|    Dmitry|    Smith|
|  4|     Елена| Сидорова|
|  5|   Алексей|   Уваров|
+---+----------+---------+



In [36]:
from pyspark.sql.functions import col

In [37]:
(
    df_customers
        .select("id", "first_name", "last_name")
        .withColumn("fn", col("first_name"))
        .show()
)

+---+----------+---------+-------+
| id|first_name|last_name|     fn|
+---+----------+---------+-------+
|  1|      Иван|   Иванов|   Иван|
|  2|      Анна|  Петрова|   Анна|
|  3|    Dmitry|    Smith| Dmitry|
|  4|     Елена| Сидорова|  Елена|
|  5|   Алексей|   Уваров|Алексей|
+---+----------+---------+-------+



In [38]:
df_customers.printSchema()

root
 |-- id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- phone_numbers: struct (nullable = true)
 |    |-- mobile: string (nullable = true)
 |    |-- home: string (nullable = true)
 |-- is_customer: boolean (nullable = true)
 |-- amount_paid: double (nullable = true)
 |-- customer_from: timestamp (nullable = true)
 |-- last_updated_ts: timestamp (nullable = true)
 |-- courses: array (nullable = true)
 |    |-- element: long (containsNull = true)



In [43]:
from pyspark.sql.functions import concat, lit, size

df_customers.select("id",
                    concat(col("first_name"), lit(" "), col("last_name")).alias("fullname"), "courses") \
                    .withColumn("courses_size", size("courses")) \
                    .show()

+---+--------------+---------+------------+
| id|      fullname|  courses|courses_size|
+---+--------------+---------+------------+
|  1|   Иван Иванов|[1, 2, 3]|           3|
|  2|  Анна Петрова|   [1, 3]|           2|
|  3|  Dmitry Smith|   [1, 2]|           2|
|  4|Елена Сидорова|      [1]|           1|
|  5|Алексей Уваров|[1, 2, 3]|           3|
+---+--------------+---------+------------+



In [45]:
(
    df_customers.select("id", "first_name", "last_name")
                .withColumnRenamed("id", "cs_id")
                .withColumnRenamed("first_name", "cs_first_name")
                .withColumnRenamed("last_name", "cs_last_name")
                .show()
)

+-----+-------------+------------+
|cs_id|cs_first_name|cs_last_name|
+-----+-------------+------------+
|    1|         Иван|      Иванов|
|    2|         Анна|     Петрова|
|    3|       Dmitry|       Smith|
|    4|        Елена|    Сидорова|
|    5|      Алексей|      Уваров|
+-----+-------------+------------+



In [46]:
(
    df_customers.select(
        col("id").alias("cs_id"),
        col("first_name").alias("cs_first_name"),
        col("last_name").alias("cs_last_name")
    )
    .show()
)

+-----+-------------+------------+
|cs_id|cs_first_name|cs_last_name|
+-----+-------------+------------+
|    1|         Иван|      Иванов|
|    2|         Анна|     Петрова|
|    3|       Dmitry|       Smith|
|    4|        Елена|    Сидорова|
|    5|      Алексей|      Уваров|
+-----+-------------+------------+



In [47]:
required_columns = ["id", "first_name", "last_name"]
target_columns = ["cs_id", "cs_first_name", "cs_last_name"]

In [48]:
df_customers.select(required_columns).show()

+---+----------+---------+
| id|first_name|last_name|
+---+----------+---------+
|  1|      Иван|   Иванов|
|  2|      Анна|  Петрова|
|  3|    Dmitry|    Smith|
|  4|     Елена| Сидорова|
|  5|   Алексей|   Уваров|
+---+----------+---------+



In [49]:
(
    df_customers
        .select(required_columns)
        .toDF(*target_columns)
        .show()
)

+-----+-------------+------------+
|cs_id|cs_first_name|cs_last_name|
+-----+-------------+------------+
|    1|         Иван|      Иванов|
|    2|         Анна|     Петрова|
|    3|       Dmitry|       Smith|
|    4|        Елена|    Сидорова|
|    5|      Алексей|      Уваров|
+-----+-------------+------------+



In [50]:
spark.stop()